# 长江中下游高温延伸期预测研究——项目规划书

本 notebook 记录研究闭环与执行顺序；完整的**项目规划书**（目标、流程、分工、概念解释）
按板块拆分在 `notebooks/项目规划书/` 目录下：

| 文件 | 板块 |
| --- | --- |
| [00_总览与目录.md](项目规划书/00_总览与目录.md) | 项目简介、关键信息速览、文档地图 |
| [01_研究背景与目标.md](项目规划书/01_研究背景与目标.md) | 背景、科学问题、总目标/分目标、创新点、非目标 |
| [02_技术路线与总体流程.md](项目规划书/02_技术路线与总体流程.md) | 技术路线图、12 步流程、阶段 P0—P6 与里程碑 M0—M6 |
| [03_数据与变量方案.md](项目规划书/03_数据与变量方案.md) | ERA5 变量清单、日统计口径、10 项质控、因子—物理假设表 |
| [04_预测目标与样本库构建.md](项目规划书/04_预测目标与样本库构建.md) | Tmax→区域平均→气候态→H、提前期定义、样本表、防泄漏五条铁律 |
| [05_模型体系与实验设计.md](项目规划书/05_模型体系与实验设计.md) | 基线到机器学习、训练/验证/测试、滚动验证、消融实验、模型冻结 |
| [06_评价体系与极端高温二阶段.md](项目规划书/06_评价体系与极端高温二阶段.md) | MAE/RMSE/R/R²、技巧评分、极端阈值、Precision/Recall/F1/HSS |
| [07_可解释性与物理机制验证.md](项目规划书/07_可解释性与物理机制验证.md) | SHAP、Permutation、合成分析、环流诊断、结论表述规范 |
| [08_分工与进度安排.md](项目规划书/08_分工与进度安排.md) | 5 人角色职责、RACI 分工矩阵、协作接口、12 个月进度 |
| [09_概念词典.md](项目规划书/09_概念词典.md) | **全部特殊概念**逐条解释（A—J 九类 + 答辩速查） |
| [10_风险清单与质量保障.md](项目规划书/10_风险清单与质量保障.md) | 风险登记表、阶段 QA 清单、代码与学术规范 |

同主题的**大挑（挑战杯）获奖项目案例集**见 [`../examples/`](../examples/README.md)：9 个可核查案例 + 六条获奖规律 + 官方项目库 4760 条记录的关键词扫描。

---


文献调研 → 建立候选预测因子库。
先阅读长江中下游高温、延伸期预测、陆气相互作用、西太副高等论文，不是看到什么变量就全部塞进去，而是先形成物理假设。例如：“前期土壤偏干 → 蒸散减弱 → 感热增强 → 后期高温增强”；“Z500异常 → 副高/高压异常 → 下沉增温 → 高温增强”。然后得到候选因子：前期温度、高温指数、土壤湿度、降水、辐射、潜热、Z500、850 hPa 风场、SST 等。
获取数据 → 构建统一样本库。
下载 ERA5 等数据，进行时间、单位、空间范围、异常值和缺测检查；然后分别构建提前第1、2、3、4周的样本。尤其注意：预测第 k 周，只能使用发布预测时已经知道的数据，不能让未来信息进入模型。你们方案目前已经把这一点定义得比较规范。
建立基线模型 + 机器学习模型。
不建议一上来只做一个 XGBoost。可以设置“气候态 → 持续性 → Linear/Ridge → Random Forest → XGBoost”的梯度。这样以后答辩时才能回答：“机器学习到底比简单方法提高了多少？”你们优化版也已经采用了这种从简单到复杂的模型体系。

训练/验证 → 筛选预测因子 + 调模型参数。
这一部分才对应你现在说的“输入预测因子、和真实值对比、反复调试”。但应该把“反复调试”限制在训练集和验证集。例如先做：
A：前期温度 → B：+土壤湿度/降水 → C：+辐射/潜热 → D：+Z500/风场 → E：+SST，看每加入一类变量，预测技巧增加多少。你们方案里已经设计了这种“分组消融实验”，它比单纯看变量重要性更加有说服力。

这里同时可以用 SHAP + permutation importance + 消融实验 判断因子的贡献，但不要简单说“SHAP最高的就是最重要的物理因子”。多个高度相关的气象变量会互相分摊贡献，因此最终判断“关键预测因子”要综合模型性能和物理机制。

冻结模型 → 独立年份最终检验。
比如按你们方案中的思路：1981—2015训练、2016—2020验证，等预测因子、模型结构、超参数都确定以后，彻底冻结模型，然后一次性预测2021—2025，和真实情况对比。测试结果不好，也不能回来根据2021—2025重新挑因子，否则测试集就失去意义。
二次处理 → 专门增强“极端高温识别”。
这一点我反而建议你们保留，因为它很可能成为项目比较好讲的特色。第一阶段模型解决的是：
“未来第1—4周到底偏不偏热、偏热多少？”

文献调研与物理假设
↓
构建候选预测因子库
↓
ERA5等多源数据获取与预处理
↓
构建提前1—4周预测样本
↓
基线模型 + RF/XGBoost建模
↓
滚动验证 + 消融实验 + SHAP筛选关键因子
↓
确定预测因子及其相对贡献，冻结模型
↓
独立年份历史回报检验
↓
极端高温二阶段识别/预警增强
↓
物理机制诊断与模型解释